# MNIST Deep Neural Network

Handwritten digit classifier built from scratch in NumPy — no high-level ML frameworks.

**Architecture:** `784 inputs → conv layer (3×3, 16 kernels) → tanh → dropout → softmax → 10 classes`

**Techniques:** Convolutional layer · Dropout regularisation · Mini-batch gradient descent · Tanh + Softmax activations

First Step: Getting the data: 
For this we are going to use keras the api for tensor flow to get our dataset. 

In [7]:
# import dataset
from keras.datasets import mnist

# import numpy and matplotlib 
import numpy as np
import matplotlib.pyplot as plt

In [19]:
(x_train, y_train), (x_test, y_test) = mnist.load_data("mnist.npz")

training_image_count, training_image_height, training_image_width = x_train.shape[0], x_train.shape[1], x_train.shape[2]
training_label_count, training_label_dtype = y_train.shape[0], y_train.dtype
print(f"Training Data:")
print(f"Training Images: {training_image_count}")
print(f"Each image is {training_image_height} x {training_image_width} pixels; totaling {training_image_height * training_image_width} pixels")
print(f"Training Labels: {training_label_count}")
print(f"Each label is a single integer (0–9), dtype: {training_label_dtype}")

test_image_count, test_image_height, test_image_width = x_test.shape[0], x_test.shape[1], x_test.shape[2]
test_label_count, test_label_dtype = y_test.shape[0], y_test.dtype
print(f"\nTest Data:")
print(f"Test Images: {test_image_count}")
print(f"Each image is {test_image_height} x {test_image_width} pixels; totaling {test_image_height * test_image_width} pixels")
print(f"Test Labels: {test_label_count}")
print(f"Each label is a single integer (0–9), dtype: {test_label_dtype}")

Training Data:
Training Images: 60000
Each image is 28 x 28 pixels; totaling 784 pixels
Training Labels: 60000
Each label is a single integer (0–9), dtype: uint8

Test Data:
Test Images: 10000
Each image is 28 x 28 pixels; totaling 784 pixels
Test Labels: 10000
Each label is a single integer (0–9), dtype: uint8


In [22]:
# function to see an image and its label
def visualize_image_and_label(index, train=True):

    # pick the right dataset based on the train flag
    if train:
        image = x_train[index]   # shape: (28, 28)
        label = y_train[index]   # single integer 0-9
        split = "Train"
    else:
        image = x_test[index]
        label = y_test[index]
        split = "Test"

    # create a figure — figsize is in inches (width, height)
    plt.figure(figsize=(3, 3))

    # imshow displays a 2D array as an image
    # cmap='gray' makes it black and white
    plt.imshow(image, cmap='gray')

    # title shows the label above the image
    plt.title(f"{split} Image {index} — Label: {label}")

    # turn off the x/y axis ticks — they're not useful for images
    plt.axis('off')

    plt.show()

Creating the network parameters 



In [25]:
# ── Hyperparameters ───────────────────────────────────────────────────────
alpha      = 2
input_size = training_image_height * training_image_width # 784
hidden_size = 100
output_size = 10
iterations = 100

## Activation Functions

**Hidden layer — Tanh**

Tanh squashes any value into the range (−1, 1) and is centred at zero. This is better than ReLU for image data because it produces both positive and negative activations, giving the network a richer signal to learn from. Centring at zero also means gradients stay larger during backpropagation, so the network learns faster.

**Output layer — Softmax**

Softmax is used because we are classifying into exactly one of 10 mutually exclusive categories (digits 0–9). It converts the 10 raw output scores into probabilities that sum to 1 — so the network outputs something like "72% sure this is a 3, 15% sure it's an 8" etc. The digit with the highest probability is taken as the prediction.

In [24]:
def tanh(x):        return np.tanh(x)
def tanh_deriv(o):  return 1 - o ** 2

def softmax(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)

## Network Sizes and Weight Shapes

Each image is a 28×28 pixel grid, flattened into a row of **784 numbers** — this is `layer_0`.

The network has two weight matrices:

| Weight matrix | Shape | Why |
|---|---|---|
| `weights_0_1` | (784, 100) | Each of the 784 pixels connects to each of the 100 hidden neurons — one weight per connection |
| `weights_1_2` | (100, 10) | Each of the 100 hidden neurons connects to each of the 10 output neurons (one per digit) |

The shape rule is always `(previous layer size, next layer size)`.

When we compute `layer_0.dot(weights_0_1)` the dimensions are `(1, 784) × (784, 100) = (1, 100)` — the 784s cancel out and we get a row of 100 hidden values. The same pattern repeats for every layer.

Remember: matrices are written as **(rows × columns)**. For two matrices to multiply, the inner dimensions must match — the result takes the outer dimensions.

---

## Initialising Weights with Random Numbers

Weights are initialised randomly using `np.random.random`, which returns values between **0.0 and 1.0**. We shift and scale this range so weights start small and centred near zero — large starting weights cause the network to diverge early in training.

```python
# Range 0.0 → 1.0  (default, too large for weights)
np.random.random((784, 100))

# Range -0.1 → 0.1  (scaled and centred at zero)
np.random.random((784, 100)) * 0.2 - 0.1

# Range -0.01 → 0.01  (narrow range, used for tanh layers)
np.random.random((784, 100)) * 0.02 - 0.01
```

The formula to shift any range is: `random × (max - min) + min`

`np.random.seed(1)` is set before initialising weights so the same random values are generated every run — this makes results reproducible and easier to debug.

In [ ]:
# set the random seed to initalize weights randomly. 
np.random.seed(1)

weights_0_1 = np.random.random((input_size, hidden_size)) * 0.02 - 0.01 # this gives a random matrix (784 x 100) of values -0.01 and 0.01
weights_0_1 = np.random.random((hidden_size, output_size)) * 0.2 - 0.1 # this gives a random matrix (100 x 10) of values -0.1, 0.1